# 2-1. 문서 로더와 청킹 전략

⏱ **소요시간**: 45분

## 학습 목표
- 금융권 현장에서 자주 마주치는 PDF / DOCX / HWPX 문서를 각각 로드할 수 있다.
- PyMuPDF, Unstructured, Upstage Layout 로더의 특성과 권장 사용 시나리오를 설명할 수 있다.
- `RecursiveCharacterTextSplitter`, `SemanticChunker`, `MarkdownHeaderTextSplitter`, `TokenTextSplitter`를 실제 데이터에 적용하고 분포를 비교하여 적절한 전략을 고를 수 있다.
- 한국어 문장 분리기 `kss`, `kiwi`를 활용해 청크 품질을 높일 수 있다.

## 핵심 키워드
`PyMuPDF` `Unstructured` `Upstage Layout` `DOCX` `HWPX` `RecursiveCharacterTextSplitter` `SemanticChunker` `MarkdownHeaderTextSplitter` `TokenTextSplitter` `kss` `kiwi`

> 🔒 모든 로더와 스플리터는 폐쇄망 환경에서 완전 동작하는 공개 라이브러리만 사용합니다. (Upstage Layout은 ☁️ API 예시로만 언급)

In [ ]:
import sys; sys.path.insert(0, '../..')
from common import get_llm, get_embeddings, provider_badge
print(provider_badge())

## 1. PDF 로더 비교 (PyMuPDF vs Unstructured vs Upstage Layout)

| 로더 | 폐쇄망 | 속도 | 레이아웃 보존 | 표/이미지 | 권장 시나리오 |
|---|---|---|---|---|---|
| PyMuPDF (`PyMuPDFLoader`) | 🔒 완전 오프라인 | 매우 빠름 | 약함 (텍스트 순서 깨질 수 있음) | 텍스트만 | 약관/지침서 등 단순 텍스트 PDF 대량 처리 |
| Unstructured (`UnstructuredPDFLoader`) | 🔒 로컬 설치 가능 | 느림 | 중간 (title, list, table 구조 태깅) | 표 일부 추출 | 계약서같이 구조가 있는 문서 |
| Upstage Layout (`UpstageLayoutAnalysisLoader`) | ☁️ **API 트래픽 있음** | 서버에 따름 | 최상 (복잡한 표/이미지 정확) | 우수 | **폐쇄망 외부 변환 서버가 허용된 경우만** 고려 |

금융권 현장에서는 대부분 **PyMuPDF로 1차 수집 → 표 많은 문서만 Unstructured로 재처리** 전략을 취하는 것이 실용적이다.

In [ ]:
from pathlib import Path

PDF_DIR = Path('../../data/pdf')
pdf_files = list(PDF_DIR.glob('*.pdf'))
print(f'발견된 PDF 파일: {len(pdf_files)}개')
for p in pdf_files:
    print(' -', p.name)

# PDF가 없으면 실습용 샘플을 즉석에서 생성 (폐쇄망 실습 편의)
if not pdf_files:
    print('\n⚠️ PDF가 없습니다. 간단한 샘플 PDF를 생성합니다.')
    import fitz  # PyMuPDF
    PDF_DIR.mkdir(parents=True, exist_ok=True)
    sample_path = PDF_DIR / 'sample_efinance.pdf'
    doc = fitz.open()
    text_pages = [
        '전자금융거래 표준약관\n\n제1조(목적) 본 약관은 전자금융거래에 관한 사항을 정함을 목적으로 한다.',
        '제5조(청약철회) 이용자는 계약 체결 후 14일 이내에 서면으로 청약을 철회할 수 있다.',
        '제10조(분실신고) 분실 · 도난 시 고객은 즉시 고객센터 또는 영업점에 신고하여야 하며, 선의무과실 없는 접근매체 노출에 대해 본인이 담보한다.',
    ]
    for txt in text_pages:
        page = doc.new_page()
        page.insert_text((50, 72), txt, fontsize=11)
    doc.save(str(sample_path))
    doc.close()
    pdf_files = [sample_path]
    print(f'✅ 샘플 생성: {sample_path}')

In [ ]:
# 1) PyMuPDF 로더 🔒
import time
from langchain_community.document_loaders import PyMuPDFLoader

target = pdf_files[0]
t0 = time.time()
pymupdf_docs = PyMuPDFLoader(str(target)).load()
print(f'[PyMuPDF] 페이지 수={len(pymupdf_docs)}, 소요시간={time.time()-t0:.3f}s')
print('---\n', pymupdf_docs[0].page_content[:300])
print('\nmetadata:', {k: v for k, v in pymupdf_docs[0].metadata.items() if k in ('source', 'page', 'total_pages')})

In [ ]:
# 2) Unstructured 로더 🔒 (구조 정보 보존)
from langchain_community.document_loaders import UnstructuredPDFLoader

t0 = time.time()
try:
    unstructured_docs = UnstructuredPDFLoader(str(target), mode='elements').load()
    print(f'[Unstructured] element 수={len(unstructured_docs)}, 소요시간={time.time()-t0:.3f}s')
    for d in unstructured_docs[:5]:
        cat = d.metadata.get('category', 'Text')
        print(f'  - [{cat}] {d.page_content[:60]}')
except Exception as e:
    print(f'⚠️ Unstructured 로드 실패 (추가 의존성 필요할 수 있음): {e}')

> ☁️ **Upstage Layout 참고** — 외부 API 적용이 허용된 환경에서는 아래처럼 사용한다.
> 
> ```python
> # from langchain_upstage import UpstageLayoutAnalysisLoader
> # loader = UpstageLayoutAnalysisLoader('contract.pdf', output_type='html')
> # docs = loader.load()
> ```
> 테이블/해더/이미지 감지가 특히 강하지만, **API 키가 외부로 전송**되므로 폐쇄망 규정 확인이 필수이다.

## 2. DOCX / HWPX 로드 데모

HWP(구버전, 바이너리)는 오픈소스 파서가 불안정하므로 **사용자 환경에서 한컴옵스 PDF로 변환 후 `load_pdf`**를 권장한다. HWPX(표준 OOXML-유사)는 `python-hwpx` 로 직접 파싱한다.

In [ ]:
from common.loaders import load_any, load_pdf, load_docx, load_hwpx
import docx  # python-docx

# 실습용 샘플 DOCX 생성
docx_path = Path('../../data/pdf/sample_compliance.docx')
d = docx.Document()
d.add_heading('컴플라이언스 안내서', 0)
d.add_paragraph('본 안내서는 내부 임직원을 대상으로 한다.')
d.add_paragraph('제1장 개인정보 처리')
d.add_paragraph('고객의 주민등록번호는 법적으로 필요한 경우에만 수집하여야 한다.')
d.save(str(docx_path))

docx_docs = load_docx(docx_path)
print('[DOCX]', docx_docs[0].page_content[:120].replace('\n', ' | '))

In [ ]:
# HWPX 데모 — 수강생이 .hwpx 파일을 놓은 경우만 실행됨
from pathlib import Path

hwpx_candidates = list(Path('../../data/pdf').glob('*.hwpx'))
if hwpx_candidates:
    hdoc = load_hwpx(hwpx_candidates[0])
    print('[HWPX]', hdoc[0].page_content[:200])
else:
    print('📝 .hwpx 파일이 없습니다. data/pdf/ 아래에 직접 배치 후 다시 실행해 보세요.')
    print('   — HWP(구버전) 파일은 `한컴오피스`에서 PDF로 저장 후 load_pdf() 사용 권장')

In [ ]:
# 확장자 자동 디스패치 헬퍼
docs_txt = load_any('../../data/corpus_ko.txt')
print(f'[load_any TXT] {len(docs_txt)} docs, len={len(docs_txt[0].page_content)} chars')

## 3. 청크 전략 비교

동일한 `corpus_ko.txt`에 네 가지 스플리터를 적용하고, 청크 길이 분포를 비교한다.

| 스플리터 | 기준 | 장점 | 단점 |
|---|---|---|---|
| `RecursiveCharacterTextSplitter` | 구두점/구분부호 기준 | 빠르고 안정적 | 의미 경계 가끔 무시 |
| `SemanticChunker` | 문장간 임베딩 유사도 | 의미 단위 분할 | 임베딩 호출 비용 |
| `MarkdownHeaderTextSplitter` | `#`/`##` 헤더 | 구조적 문서에 이상적 | 일반 PDF에 맞지 않음 |
| `TokenTextSplitter` | tiktoken 토큰 수 | LLM 컨텍스트 정확 | 문장 중간 절단 |

In [ ]:
# 코퍼스 로드
from langchain_core.documents import Document

corpus_path = Path('../../data/corpus_ko.txt')
corpus_text = corpus_path.read_text(encoding='utf-8')
print(f'전체 글자 수: {len(corpus_text)}')
corpus_doc = Document(page_content=corpus_text, metadata={'source': str(corpus_path)})

In [ ]:
# 3-1) RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=['\n\n', '\n', '. ', '! ', '? ', ' ', ''],
)
recursive_chunks = recursive_splitter.split_documents([corpus_doc])
print(f'[Recursive] chunks={len(recursive_chunks)}, avg_len={sum(len(c.page_content) for c in recursive_chunks)/len(recursive_chunks):.1f}')

In [ ]:
# 3-2) TokenTextSplitter (tiktoken 기반) — LLM 컨텍스트 예산과 직결됨
from langchain_text_splitters import TokenTextSplitter

token_splitter = TokenTextSplitter(chunk_size=120, chunk_overlap=20, encoding_name='cl100k_base')
token_chunks = token_splitter.split_documents([corpus_doc])
print(f'[Token] chunks={len(token_chunks)}, avg_len(char)={sum(len(c.page_content) for c in token_chunks)/len(token_chunks):.1f}')

In [ ]:
# 3-3) MarkdownHeaderTextSplitter — 헤더 기반
from langchain_text_splitters import MarkdownHeaderTextSplitter

md_text = '''# 전자금융거래
## 정의
전자적 장치를 통해 자동화된 방식으로 금융서비스를 이용하는 거래를 말한다.
## 청약철회
이용자는 14일 이내 서면으로 청약을 철회할 수 있다.
# 분실신고
## 절차
즉시 고객센터 또는 영업점에 신고한다.
'''
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[('#', 'h1'), ('##', 'h2')])
md_chunks = md_splitter.split_text(md_text)
for c in md_chunks:
    print(c.metadata, '->', c.page_content[:40])

In [ ]:
# 3-4) SemanticChunker — 임베딩 거리 기반 (비용 있음)
from langchain_experimental.text_splitter import SemanticChunker

try:
    semantic_splitter = SemanticChunker(
        embeddings=get_embeddings(),
        breakpoint_threshold_type='percentile',
        breakpoint_threshold_amount=85,
    )
    semantic_chunks = semantic_splitter.split_documents([corpus_doc])
    print(f'[Semantic] chunks={len(semantic_chunks)}, avg_len={sum(len(c.page_content) for c in semantic_chunks)/len(semantic_chunks):.1f}')
    for i, c in enumerate(semantic_chunks[:3]):
        print(f'  [{i}] {c.page_content[:80]}…')
except Exception as e:
    print(f'⚠️ SemanticChunker 실패 (임베딩 provider 확인): {e}')
    semantic_chunks = []

## 4. 한국어 문장 분리기: kss & kiwi

`RecursiveCharacterTextSplitter`가 `. ` 을 사용하는데, 한국어는 종결 형태(`다.`, `으세요.`, `합니다.`)가 붙으므로 **`합니다.그`같은 경계를 잘못 끊는 경우**가 많다. `kss`, `kiwi`를 써서 사전 분절하면 청크 품질이 개선된다.

In [ ]:
# kss 사용
import kss

sample = '전자금융거래는 자동화된 방식이다. 이용자는 14일 이내 철회할 수 있다. 분실시 즉시 신고한다!'
sentences_kss = kss.split_sentences(sample)
print('[kss]', sentences_kss)

# kiwi 사용
from kiwipiepy import Kiwi
kiwi = Kiwi()
sentences_kiwi = [s.text for s in kiwi.split_into_sents(sample)]
print('[kiwi]', sentences_kiwi)

In [ ]:
# 한국어 문장 단위 추출 후 recursive로 합치기 (실전 패턴)
def korean_aware_chunks(text: str, chunk_size: int = 300, overlap: int = 40):
    sents = [s.text.strip() for s in kiwi.split_into_sents(text) if s.text.strip()]
    chunks, buf = [], ''
    for s in sents:
        if len(buf) + len(s) + 1 <= chunk_size:
            buf = (buf + ' ' + s).strip()
        else:
            if buf:
                chunks.append(buf)
            # 오버랩 보존: 직전 chunk의 끝부분을 잃더라도 새 chunk는 단순 새문장부터
            buf = (buf[-overlap:] + ' ' + s).strip() if buf else s
    if buf:
        chunks.append(buf)
    return chunks

ko_chunks = korean_aware_chunks(corpus_text, chunk_size=300, overlap=40)
print(f'[Korean-aware] chunks={len(ko_chunks)}, avg_len={sum(len(c) for c in ko_chunks)/len(ko_chunks):.1f}')
for c in ko_chunks[:2]:
    print(' -', c[:80], '…')

## 5. 청크 길이 분포 시각화

In [ ]:
import matplotlib.pyplot as plt

all_strategies = {
    'Recursive': [len(c.page_content) for c in recursive_chunks],
    'Token': [len(c.page_content) for c in token_chunks],
    'Korean-aware': [len(c) for c in ko_chunks],
}
if semantic_chunks:
    all_strategies['Semantic'] = [len(c.page_content) for c in semantic_chunks]

fig, ax = plt.subplots(figsize=(8, 4))
for name, lens in all_strategies.items():
    ax.hist(lens, bins=12, alpha=0.5, label=f'{name} (n={len(lens)})')
ax.set_xlabel('chunk length (chars)')
ax.set_ylabel('count')
ax.set_title('Chunk length distribution by splitter')
ax.legend()
plt.tight_layout()
plt.show()

## 정리

- 로더는 **다양한 소스를 동일한 `Document` 형식으로** 평준화한다.
- 청크 크기는 **임베딩 입력 생성 비용 ↔ 의미 보존**의 트레이드오프. 300자 안팎, 50자 overlap이 한국어 약관 문서에 무난.
- `SemanticChunker`는 일회성 토큰 비용이 든다(최초 1회 인덱싱시). 업데이트가 적은 금융 약관에 적합.
- 한국어는 **`kiwi.split_into_sents`로 먼저 문장 분리 → 청크 병합**이 실전에서 잘 동작.

## 과제
1. 자사 내부 규정 PDF 1개를 `data/pdf/`에 넣고 PyMuPDF와 Unstructured 모두로 로드해 출력을 눈으로 비교하세요. 표가 많은 문서에서 차이가 보이나요?
2. `chunk_size`를 100 / 500 / 1000으로 바꿔 Recursive로 분할 후 분포를 시각화해보세요.
3. `korean_aware_chunks`를 개선해서 **문장이 너무 짧으면 이웃과 병합**하는 로직을 추가해보세요.

## 더 읽어보기
- teddylee777/langchain-kr — [06-DocumentLoader](https://github.com/teddylee777/langchain-kr/tree/main/06-DocumentLoader)
- teddylee777/langchain-kr — [07-TextSplitter](https://github.com/teddylee777/langchain-kr/tree/main/07-TextSplitter)
- [LangChain Text Splitters 공식 문서](https://python.langchain.com/docs/concepts/text_splitters/)
- [Kiwi 형태소 분석기](https://github.com/bab2min/Kiwi) / [KSS Docs](https://github.com/hyunwoongko/kss)